In [ ]:
import os
import yaml
import shutil

# === Input folder ===
root = r"F:\0Defense\Dataset\Vegetables_Dataset_seperate_folder_increase_size"

# === Output folder ===
output = r"F:\0Defense\Dataset\merged_dataset"

vegetables = [
    "Bean", "Cabbage", "Cauliflower", "Chilli",
    "Cucumber", "Eggplant", "Onion", "Potato", "Tomato"
]

splits = ["train", "valid", "test"]

# Create final folder structure
for split in splits:
    os.makedirs(os.path.join(output, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output, "labels", split), exist_ok=True)

global_class_names = []
class_offset = 0

print("\n====== STARTING MERGE & LABEL REINDEXING ======\n")

for veg in vegetables:
    print(f"\nProcessing: {veg}")

    yaml_path = os.path.join(root, veg, "data.yaml")
    with open(yaml_path, "r") as f:
        data_yaml = yaml.safe_load(f)

    local_names = data_yaml["names"]
    local_nc = data_yaml["nc"]

    # Assign global class IDs
    new_class_map = {i: i + class_offset for i in range(local_nc)}

    # Add class names with vegetable prefix
    for cname in local_names:
        global_class_names.append(f"{veg.lower()}_{cname}")

    print(f"  Local classes: {local_nc}")
    print(f"  Global class offset: {class_offset}")

    # Reindex labels
    for split in splits:

        img_src = os.path.join(root, veg, split, "images")
        lbl_src = os.path.join(root, veg, split, "labels")

        for fname in os.listdir(img_src):
            shutil.copy(
                os.path.join(img_src, fname),
                os.path.join(output, "images", split, f"{veg}_{fname}")
            )

        for fname in os.listdir(lbl_src):
            old_label = os.path.join(lbl_src, fname)
            new_label = os.path.join(output, "labels", split, f"{veg}_{fname}")

            with open(old_label, "r") as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                old_class = int(parts[0])
                parts[0] = str(new_class_map[old_class])
                new_lines.append(" ".join(parts) + "\n")

            with open(new_label, "w") as f:
                f.writelines(new_lines)

    class_offset += local_nc

print("\n====== MERGE COMPLETE ======\n")

# Create final dataset.yaml
final_yaml = {
    "train": f"{output.replace(os.sep, '/')}/images/train",
    "val": f"{output.replace(os.sep, '/')}/images/valid",
    "test": f"{output.replace(os.sep, '/')}/images/test",
    "nc": len(global_class_names),
    "names": global_class_names
}

yaml_path = os.path.join(output, "dataset.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(final_yaml, f)

print("Final dataset.yaml created!")
print(f"Total classes = {len(global_class_names)}")
print("\nDone!")
